# Laboratorio 2

## Integrantes

- Sergio Orellana 221122
- Ricardo Chuy 221007
- Rodrigo Mansilla 22611

Una empresa de gestión de infraestructura hospitalaria opera un sistema de ascensores en un edificio de cinco pisos. En horas críticas, el tiempo de espera de camillas y personal médico tiene consecuencias directas sobre la atención al paciente. Por ello, la gerencia desea optimizar la política de despacho mediante Programación Dinámica antes de invertir en un sistema de aprendizaje completo.

Su grupo ha sido contratado para modelar el problema como un MDP, implementar Policy Iteration y Value Iteration desde cero, comparar su comportamiento, y producir un dictamen técnico con recomendaciones concretas para la gerencia.

## Task 1

### Dado el contexto del sistema de ascensores, diseñen formalmente el MDP que representa el problema. El diseño debe especificar con precisión y justificación:

#### 1. El espacio de estados $\mathcal{S}$: ¿qué variables definen completamente la situación del sistema en un instante dado? Justifiquen cada variable incluida y cada variable omitida. Para cada variable omitida, argumenten qué supuesto están haciendo y qué consecuencia tiene ese supuesto sobre la validez del modelo.

_Respuesta:_

Modelo un ascensor representativo mediante el estado:

$$s=(p,o,\mathbf{q},\mathbf{c})$$

donde:

- $p \in \{1,2,3,4,5\}$ representa el piso actual del ascensor.
- $o \in \{0,1\}$ representa el estado de la puerta: cerrada o abierta.
- $\mathbf{q}=(q_1,\ldots,q_5)$ representa las solicitudes externas de cada piso.
- Cada $q_i \in \{0,1,2,3,4\}$ significa, respectivamente: sin solicitud, solicitud normal reciente, solicitud normal demorada, solicitud de emergencia reciente o solicitud de emergencia demorada.
- $\mathbf{c}=(c_1,\ldots,c_5)$ representa los destinos seleccionados dentro de la cabina, donde $c_i \in \{0,1\}$.

Incluyo el piso porque determina qué movimientos son posibles. Asimismo, incluyo el estado de la puerta porque el ascensor no debe desplazarse mientras permanece abierta. Además, las solicitudes externas y los destinos internos permiten conocer tanto a quién debe recoger como a dónde debe transportar a los usuarios.

Uso categorías de antigüedad en $\mathbf{q}$ porque dos solicitudes del mismo tipo no deberían recibir la misma prioridad cuando una lleva más tiempo esperando. Por consiguiente, esta representación aproxima la propiedad de Markov, ya que la decisión futura depende de la situación resumida en el estado actual y no de toda la secuencia histórica.

Omito la cantidad exacta de personas, su peso individual y la ocupación detallada. Por lo que, supongo una carga promedio; sin embargo, esta simplificación puede producir una estimación imprecisa del consumo energético y de la capacidad disponible. También omito fallas mecánicas, mantenimiento, tiempos exactos de apertura de puertas y la coordinación entre varios ascensores. Por lo tanto, el modelo es útil para analizar una cabina representativa, pero tendría que ampliarse antes de controlar una flota real.

Con esta representación, el número de estados es:

$$|\mathcal{S}|=5 \times 2 \times 5^5 \times 2^5=1{,}000{,}000$$


#### 2. El espacio de acciones $\mathcal{A}$: definan las acciones disponibles para el sistema de control. Argumenten si el espacio debe ser discreto o continuo para este dominio, y si existen acciones que deben restringirse en ciertos estados y cómo se modela esa restricción.

_Respuesta:_

Defino el espacio de acciones como:

$$\mathcal{A}=\{\text{subir},\text{bajar},\text{atender},\text{esperar}\}$$

Considero que el espacio es discreto porque el controlador toma decisiones operacionales separadas en cada instante. La acción subir desplaza el ascensor un piso hacia arriba, mientras que bajar lo desplaza un piso hacia abajo. Por otra parte, atender abre la puerta y procesa la solicitud o el destino del piso actual. Finalmente, esperar mantiene la posición y permite cerrar la puerta o aguardar una nueva solicitud.

Restrinjo subir en el piso 5 y bajar en el piso 1. Asimismo, prohíbo los movimientos cuando la puerta está abierta. Para conservar un espacio de acciones uniforme en todos los estados, modelo cada acción inválida como una transición al mismo estado con una penalización. De esta manera, el MDP mantiene cuatro acciones formalmente disponibles, pero aprende que las acciones incompatibles con la seguridad producen un resultado desfavorable.


#### 3. La función de recompensa $r(s,a,s')$: diseñen una función que capture el objetivo operacional real. Consideren al menos los siguientes objetivos potencialmente conflictivos: minimizar tiempo de espera promedio, priorizar pisos de emergencia y minimizar consumo energético. Argumenten cómo ponderan esos objetivos y qué consecuencias tendría una ponderación incorrecta sobre el comportamiento del agente.

_Respuesta:_

Diseño la función de recompensa de la siguiente manera:

$$r(s,a,s')=
40I_{\mathrm{em}}+
12I_{\mathrm{normal}}-
18I_{\mathrm{em\_espera}}-
8W(s)-
2C(a)-
25I_{\mathrm{inv}}$$

donde:

- $I_{\mathrm{em}}=1$ cuando el ascensor atiende una solicitud de emergencia.
- $I_{\mathrm{normal}}=1$ cuando atiende una solicitud normal.
- $I_{\mathrm{em\_espera}}=1$ cuando permanece una emergencia pendiente después de la acción.
- $W(s)\in[0,1]$ representa la presión normalizada del tiempo de espera.
- $C(a)\in[0,1]$ representa el consumo energético relativo de la acción.
- $I_{\mathrm{inv}}=1$ cuando se intenta una acción inválida o insegura.

Asigno el mayor incentivo a la atención de emergencias porque una demora puede afectar directamente la atención clínica. Además, recompenso la atención normal, pero con un peso menor. Penalizo la espera acumulada para evitar que el ascensor ignore solicitudes antiguas y, al mismo tiempo, agrego un costo energético moderado para favorecer recorridos eficientes sin sacrificar la atención prioritaria.

La magnitud máxima absoluta de la recompensa queda acotada por:

$$R_{\max}=53$$

Una ponderación incorrecta puede producir conductas no deseadas. Por ejemplo, si el costo energético domina la función, el ascensor podría permanecer inmóvil para ahorrar energía. En cambio, si la recompensa por emergencia es excesiva y no existe una penalización por espera normal, las solicitudes comunes podrían quedar desatendidas durante demasiado tiempo. Por esta razón, mantengo una prioridad clara para emergencias, pero también penalizo la congestión general.


#### 4. La función de transición $p(s' \mid s,a)$: argumenten si el entorno es determinista o estocástico. Si es estocástico, identifiquen las fuentes de aleatoriedad y escriban al menos cuatro transiciones concretas con sus probabilidades justificadas.

_Respuesta:_

Considero que el entorno es estocástico porque las solicitudes aparecen de manera impredecible y porque el movimiento o la atención pueden sufrir retrasos por sensores, puertas, usuarios o camillas. Por consiguiente, la siguiente situación no queda determinada únicamente por la acción seleccionada.

Utilizo las siguientes probabilidades como una primera aproximación operacional:

##### Transición 1: movimiento exitoso hacia arriba

Si el ascensor está en el piso 2, la puerta está cerrada y se selecciona **subir**:

$$P(p'=3,o'=0 \mid p=2,o=0,a=\text{subir})=0.90$$

Asigno una probabilidad de $0.90$ porque normalmente el movimiento se completa sin inconvenientes.

##### Transición 2: retraso durante el movimiento

Bajo el mismo estado y la misma acción:

$$P(p'=2,o'=0 \mid p=2,o=0,a=\text{subir})=0.10$$

Este resultado representa un retraso temporal causado por sensores, reapertura de puertas o bloqueo operacional.

### Transición 3: atención exitosa de una emergencia

Si existe una solicitud de emergencia en el piso actual y se selecciona **atender**:

$$P(q'_p=0,o'=1 \mid q_p\in\{3,4\},a=\text{atender})=0.95$$

La probabilidad $0.95$ representa que la atención normalmente elimina la solicitud, aunque puede existir una demora durante el abordaje.

##### Transición 4: la emergencia continúa pendiente

En el mismo caso:

$$P(q'_p=q_p,o'=1 \mid q_p\in\{3,4\},a=\text{atender})=0.05$$

Esta probabilidad representa una camilla que todavía no ha terminado de ingresar o la puerta debe permanecer abierta.

##### Transición 5: ausencia de nuevas solicitudes

Si no existe demanda y el ascensor selecciona **esperar**:

$$P(\mathbf{q}'=\mathbf{0}\mid\mathbf{q}=\mathbf{0},a=\text{esperar})=0.70$$

##### Transición 6: aparición de una solicitud normal

Supongo que existe una probabilidad total de $0.20$ de que aparezca una solicitud normal en otro piso. Si hay cuatro pisos candidatos y todos son igualmente probables:

$$P(q'_j=1)=\frac{0.20}{4}=0.05$$



#### 5. El factor de descuento $\gamma$: propongan un valor y justifíquenlo en términos del dominio, no solo matemáticamente. ¿Cómo cambia la política óptima si $\gamma$ es cercano a 0 versus cercano a 1 en este contexto específico?

_Respuesta:_

Propongo utilizar:

$$\gamma=0.95$$

Elijo este valor porque una decisión de despacho afecta varias acciones posteriores. Por ejemplo, atender la solicitud más cercana puede ser conveniente de inmediato, pero también puede dejar al ascensor lejos de un piso de emergencia o aumentar la congestión futura. Por lo tanto, necesito valorar consecuencias de mediano y largo plazo sin otorgar el mismo peso a eventos demasiado lejanos e inciertos.

Si $\gamma$ es cercano a $0$, la política prioriza casi exclusivamente la recompensa inmediata. Por lo que, podría atender la solicitud más próxima o evitar un movimiento costoso, aunque esa decisión genere una espera elevada posteriormente.

En cambio, si $\gamma$ es cercano a $1$, la política considera con mayor intensidad la acumulación futura de esperas, emergencias y consumo energético. Esto favorece decisiones estratégicas, como posicionarse cerca de pisos críticos. Sin embargo, un valor demasiado alto aumenta el tiempo requerido para la convergencia y puede hacer que el modelo dependa excesivamente de predicciones futuras poco precisas.
